In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as fn
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName('mnm_lab') \
    .master('local[*]') \
    .getOrCreate()

current_datetime = datetime.now()

print(f'|{current_datetime}| Starting session')

|2026-03-16 18:14:16.020484| Starting session


In [3]:
file_name = 'mnm_dataset.csv'
path_to_file = rf'G:\Work\projects\test_rep\DE - 101 Lab 7.2\data\{file_name}'

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(f'{path_to_file}')

In [4]:
df_csv.show(5)

+-----+------+-----+
|State| Color|Count|
+-----+------+-----+
|   TX|   Red|   20|
|   NV|  Blue|   66|
|   CO|  Blue|   79|
|   OR|  Blue|   71|
|   WA|Yellow|   93|
+-----+------+-----+
only showing top 5 rows



In [10]:
# aggregate count of all colors and groupBy state and color
# orderBy descending order
df_csv_aggr = (df_csv.select('State', 'Color','Count')
               .groupBy('State', 'Color')
               .agg(fn.sum('Count').alias('sum_value'),
                    fn.round(fn.avg('Count'), 2).alias('avg_value'),
                    fn.count('Count').alias('rows_count'))
               .orderBy('sum_value', ascending = False)
)

df_csv_aggr.show()

+-----+------+---------+---------+----------+
|State| Color|sum_value|avg_value|rows_count|
+-----+------+---------+---------+----------+
|   CA|Yellow|   100956|    55.87|      1807|
|   WA| Green|    96486|    54.24|      1779|
|   CA| Brown|    95762|    55.74|      1718|
|   TX| Green|    95753|    55.13|      1737|
|   TX|   Red|    95404|    55.31|      1725|
|   CO|Yellow|    95038|    55.22|      1721|
|   NM|   Red|    94699|    56.03|      1690|
|   OR|Orange|    94514|    54.22|      1743|
|   WY| Green|    94339|    55.66|      1695|
|   NV|Orange|    93929|    54.87|      1712|
|   TX|Yellow|    93819|    55.09|      1703|
|   CO| Green|    93724|    54.71|      1713|
|   CO| Brown|    93692|    56.58|      1656|
|   CA| Green|    93505|    54.27|      1723|
|   NM| Brown|    93447|    55.39|      1687|
|   CO|  Blue|    93412|    55.11|      1695|
|   WA|   Red|    93332|    55.85|      1671|
|   WA| Brown|    93082|    55.77|      1669|
|   WA|Yellow|    92920|    55.87|

In [ ]:
# show all the resulting aggregation for all the dates and colors
print(f'Total Rows = {df_csv_aggr.count()}')

Total Rows = 60


In [ ]:
# find the aggregate count for California by filtering
# show the resulting aggregation for California
df_csv_aggr_ca = (df_csv_aggr.select('*')
                  .where(df_csv_aggr.State == 'CA')
)
df_csv_aggr_ca.show()

+-----+------+---------+---------+----------+
|State| Color|sum_value|avg_value|rows_count|
+-----+------+---------+---------+----------+
|   CA|Yellow|   100956|    55.87|      1807|
|   CA| Brown|    95762|    55.74|      1718|
|   CA| Green|    93505|    54.27|      1723|
|   CA|   Red|    91527|    55.27|      1656|
|   CA|Orange|    90311|     54.5|      1657|
|   CA|  Blue|    89123|     55.6|      1603|
+-----+------+---------+---------+----------+



In [8]:
spark.stop()